<a href="https://colab.research.google.com/github/patrickkirbyjr/DataPat/blob/main/running_total.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Window Functions: Calculate Running Totals in Python with DuckDB

In [22]:
#import libraries
import duckdb

In [23]:
#ingest data
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
df_preview = duckdb.sql(f"SELECT * FROM '{url}' LIMIT 5").df()
df_preview

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [25]:
#write query for trip distance running totals for January 2023
query = f"""

  WITH daily_distance AS (
    SELECT
      DATE_TRUNC('day', tpep_pickup_datetime) AS pickup_day,
      SUM(trip_distance) AS daily_total
    FROM '{url}'
    WHERE tpep_pickup_datetime >= '2023-01-01'
      AND tpep_pickup_datetime < '2023-02-01'
    GROUP BY pickup_day
  )

  SELECT
    pickup_day,
    daily_total,
    SUM(daily_total) OVER (ORDER BY pickup_day) AS running_total
  FROM daily_distance
  ORDER BY pickup_day

"""
df = duckdb.sql(query).df()
df

,pickup_day,daily_total,running_total
0,2023-01-01,395174.12,395174.12
1,2023-01-02,298874.57,694048.69
2,2023-01-03,348516.38,1042565.07
3,2023-01-04,336227.96,1378793.03
4,2023-01-05,357666.94,1736459.97
5,2023-01-06,341940.39,2078400.36
6,2023-01-07,572803.98,2651204.34
7,2023-01-08,335369.48,2986573.82
8,2023-01-09,319980.69,3306554.51
9,2023-01-10,316035.86,3622590.37
